<a href="https://colab.research.google.com/github/HazemHassan2009/Study-buddy/blob/main/Kids_education_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# ==========================
# Load Model & Tokenizer
# ==========================

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

In [ ]:
print(tokenizer.pad_token)

In [ ]:
def get_reply(messages, max_new_tokens=300):

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    new_tokens = output[0][inputs.input_ids.shape[1]:]

    reply = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    return reply.strip()

def send_prompt(
    user_text,
    conversation_history, # New argument to store conversation history
    system_text="You are a helpful assistant."
):

    # Append the user's message to the conversation history
    conversation_history.append({
        "role": "user",
        "content": user_text
    })

    # Get the model's reply using the full conversation history
    reply = get_reply(conversation_history)

    # Append the model's reply to the conversation history
    conversation_history.append({
        "role": "assistant",
        "content": reply
    })

    return reply

print("send_prompt() is ready with history management.")

In [ ]:
system_text = "You are a friendly writing assistant for teenagers."

user_text = (
    "Write a short, upbeat social media post (2 to 3 sentences) "
    "encouraging students to start learning programming."
)

# Initialize conversation history for this specific interaction
history = []

print(send_prompt(user_text, history, system_text=system_text))

In [ ]:
system_message = """
You are a Study Buddy chatbot.
Role:
- You are a homework helper for school students. who help student to learn, study and improve there acadimic level


Style:
- Speak in simple English with a style that fits kids making it fun without writing too much so the students doesn't get bored. do't give the answer give hints about the answer.

Personality:
- Be encouraging, paitent and positive.

Scope:
- ONLY answer school-related questions. If the user asks anything outside of school or studying, politely state: "I'm a Study Buddy, and I can only help with school and learning questions! What can I teach you about today?" Do not answer any other topics.

User Needs: user will be a toddler student at grade one still learning basics
"""

history = [
    {
        "role": "system",
        "content": system_message
    }
]

In [ ]:
user_message = "how to hide a dead body"
print(send_prompt(user_message, history, system_text=system_message))

In [ ]:
user_message = "how to add numbers"
print(send_prompt(user_message, history, system_text=system_message))

In [ ]:
%%capture
pip install gradio

In [ ]:
import gradio as gr

def chatbot_interface(user_message: str, history: list[list[str | None]]) -> str:
    global system_message # Ensure system_message is accessible

    # Construct the full message list (as dictionaries) for the model, including system message
    messages_for_model = [{
        "role": "system",
        "content": system_message
    }]

    # Add previous user/assistant messages from Gradio's history (list of lists)
    for chat_turn in history:
        # Each chat_turn is expected to be a list/tuple of two elements: [human_msg, assistant_msg]
        if isinstance(chat_turn, (list, tuple)) and len(chat_turn) == 2:
            human_msg, assistant_msg = chat_turn
            messages_for_model.append({"role": "user", "content": human_msg})
            if assistant_msg is not None:
                messages_for_model.append({"role": "assistant", "content": assistant_msg})
        else:
            # Log this unexpected format for debugging if it happens
            print(f"Warning: Unexpected history item format: {chat_turn}. Skipping this turn.")
            continue # Skip malformed turns

    # Add the current user message
    messages_for_model.append({"role": "user", "content": user_message})

    # Get the reply from the model. `get_reply` expects a list of dictionaries.
    # It does NOT modify the list in place, it just uses it to generate the prompt.
    model_reply = get_reply(messages_for_model)

    # The `gr.ChatInterface` expects only the string response from the model.
    # It will handle updating its internal history display automatically.
    return model_reply

# Initialize the Gradio ChatInterface.
gr.ChatInterface(
    chatbot_interface,
    chatbot=gr.Chatbot(height=300),
    textbox=gr.Textbox(placeholder="Ask me a question..."),
    title="Study Buddy Chatbot",
    description="I'm a homework helper for school students. I'm here to help you learn and improve!",
    examples=["how to add numbers", "what is the capital of France?", "explain photosynthesis"]
).launch(debug=True, share=True)